메타클래스를 사용하면 자동으로 타입을 등록할 수도 있다. 간단한 식별자를 이용해 그에 해당하는 클래스를 찾는 역검색을 하고 싶을때 이런 등록 기능이 유용하다.

In [ ]:
import json

class Serializable:
    def __init__(self, *args):
        self.args = args

    def serialize(self):
        return json.dumps({'args': self.args})


class Point2D(Serializable):
    def __init__(self, x, y):
        super().__init__(x, y)
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Point2D({self.x}, {self.y})'

point = Point2D(5, 3)
print('객체:', point)
print('직렬화한 값:', point.serialize())

`Serializable` 상속하여 역직렬화를 구현해 보자.

In [ ]:
class Deserializable(Serializable):
    @classmethod
    def deserialize(cls, json_data):
        params = json.loads(json_data)
        return cls(*params['args'])

class BetterPoint2D(Deserializable):
    def __init__(self, x, y):
        super().__init__(x, y)
        self.x = x
        self.y = y

    def __repr__(self):
        return f'BetterPoint2D({self.x}, {self.y})'

before = BetterPoint2D(5, 3)
print('이전: ', before)
data = before.serialize()
print('직렬화한 값:', data)
after = BetterPoint2D.deserialize(data)
print('이후: ', after)

이 접근 방식은 직렬화할 데이터 타입(`Point2D`, `BetterPoint2D`)을 알고 있을 경우에만 사용 가능하다는 문제점이 있다.

어떤 클래스가 오더라도 직렬화 처리를 하는 것이 이상적이다. 아래는 객체의 이름을 직렬화해 JSON 데이터에 포함시키는 방법을 이용한 것이다.

In [ ]:
class BetterSerializable:
    def __init__(self, *args):
        self.args = args

    def serialize(self):
        return json.dumps({
            'class': self.__class__.__name__,
            'args': self.args,
        })

    def __repr__(self):
        name = self.__class__.__name__
        args_str = ', '.join(str(x) for x in self.args)
        return f'{name}({args_str})'

registry = {}

def register_class(target_class):
    registry[target_class.__name__] = target_class

def deserialize(data):
    params = json.loads(data)
    name = params['class']
    target_class = registry[name]
    return target_class(*params['args'])

class EvenBetterPoint2D(BetterSerializable):
    def __init__(self, x, y):
        super().__init__(x, y)
        self.x = x
        self.y = y

# 역직렬화를 위해 클래스 등록을 해줘야 한다
register_class(EvenBetterPoint2D)

before = EvenBetterPoint2D(5, 3)
print('이전: ', before)
data = before.serialize()
print('직렬화한 값:', data)
after = deserialize(data)
print('이후: ', after)

위 방식의 문제점은 `register_class` 호출을 잊어버릴 수 있다는 점이다.

In [ ]:
class Point3D(BetterSerializable):
    def __init__(self, x, y, z):
        super().__init__(x, y, z)
        self.x = x
        self.y = y
        self.z = z

point = Point3D(5, 9, -4)
data = point.serialize()
deserialize(data)

이러한 경우에 메타클래스가 유용하게 동작한다. `BetterSerializable` 사용 의도를 감지하고 적절한 동작을 수행해 `register_class`를 호출해 준다면?

In [ ]:
class Meta(type):
    def __new__(meta, name, bases, class_dict):
        cls = type.__new__(meta, name, bases, class_dict)
        register_class(cls)
        return cls

class RegisteredSerializable(BetterSerializable,
                             metaclass=Meta):
    pass

class Vector3D(RegisteredSerializable):
    def __init__(self, x, y, z):
        super().__init__(x, y, z)
        self.x, self.y, self.z = x, y, z

before = Vector3D(10, -7, 3)
print('이전: ', before)
data = before.serialize()
print('직렬화한 값:', data)
print('이후: ', deserialize(data))

더 좋은 방법은 `__init_subclass__` 특별 메소드를 사용하는 것이다.

In [ ]:
class BetterRegisteredSerializable(BetterSerializable):
    def __init_subclass__(cls):
        super().__init_subclass__()
        register_class(cls)

class Vector1D(BetterRegisteredSerializable):
    def __init__(self, magnitude):
        super().__init__(magnitude)
        self.magnitude = magnitude

before = Vector1D(6)
print('이전: ', before)
data = before.serialize()
print('직렬화한 값:', data)
print('이후: ', deserialize(data))